## Imports an libraries

In [3]:
# Enable inline plots in the notebook
%matplotlib inline

# Import library functions needed
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.animation as animation
from IPython.display import clear_output
from IPython.display import HTML
#from IPython.core.debugger import set_trace # Activates debugging features

def rasterplot(ax, x, y, x_label, y_label):
# Function used to plot spike times
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.scatter(x, y, marker='|')
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    

# Set default figure size (Might not be needed?)
plt.rcParams['figure.figsize'] = [10, 10]


## Dataset Generator
 Dataset Generator
    
Generates a sequence of frames showing a single pixel
moving horizontally across the frame.

Parameters:
* frame_x (int): Width of frame.
* frame_y (int): Hight of frame.
* seq_length (int): Number of frames in sequence.
* initial_direction (string): "right" or "left". 
* speed (int): Pixels moved per frame.
* noise_level (float): pixel noise.
* p_right, p_left, p_stop (float): probability the dot moves right, left and stands still

Returns:
* numpy array of shape (seq_length, frame_y, frame_x)

In [11]:
def generate_motion_sequence(
    frame_x=64,     #fame_x higher since we want left to right
    frame_y=10,     #frame_y lower since we only care about horizontal movement
    seq_length=60,
    initial_direction="stop", #gets this from create_dataset
    speed=0,                #gets this from create_dataset
    noise_level=0.0,        #gets this from create_dataset
    p_right=0.0,                #get this from create_dataset
    p_left=0.0,               #get this from create_dataset     
    p_stop=0.0          #get this from create_dataset
):
    frames = []

    y_pos = np.random.randint(2, frame_y-2)

    #Start position depends on initial direction
    if initial_direction == "right":
        x = 0
    else:
        x = frame_x - 1

    for t in range(seq_length):
        frame = np.zeros((frame_y, frame_x))

        #Keep inside bounds
        x = np.clip(x, 0, frame_x - 1)

        frame[y_pos, int(x)] = 1.0

        #Add noise
        if noise_level > 0:
            noise = np.random.normal(0, noise_level, (frame_y, frame_x))
            frame += noise
            frame = np.clip(frame, 0, 1)

        frames.append(frame)

        #Probabilistic motion
        motion = np.random.choice(
            ["right", "left", "stop"],
            p=[p_right, p_left, p_stop]
        )

        if motion == "right":
            x += speed
        elif motion == "left":
            x -= speed
        elif motion == " stop":
            pass

    return np.array(frames)



def create_dataset(n_samples=1000, seed=None, noise_level=0.0, direction="right", p_right=0.5, p_left=0.5, p_stop=0.0):     #default values, can be overridden when calling the function
    if seed is not None:
        np.random.seed(seed)

    x = []      #Data sequences
    y = []      #Lables

    for _ in range(n_samples):

        seq = generate_motion_sequence(
            initial_direction=direction,
            speed=1,
            noise_level=noise_level,
            p_right=p_right,
            p_left=p_left,
            p_stop=p_stop
        )

        x.append(seq)
        y.append(direction)

    return np.array(x), np.array(y)


# Visualize the data sequence
def animate_sequence(sequence):
    fig, ax = plt.subplots()
    img = ax.imshow(sequence[0], cmap='gray', vmin=0, vmax=1)
    ax.set_axis_off()

    def update(frame):
        img.set_data(sequence[frame])
        return [img]

    ani = animation.FuncAnimation(
        fig, update, frames=len(sequence), interval=200
    )

    plt.close(fig)  # Prevent duplicate static plot
    return HTML(ani.to_jshtml())

x, y = create_dataset(n_samples=1, seed=67, noise_level=0.05, direction="right", p_right=0.6, p_left=0.2, p_stop=0.2)
animate_sequence(x[0])

## Delta Modulation (Spike Encoding)
Convert pixel changes to spikes

In [3]:
def encode_to_spikes(frames, threshold=0.1):
    """
    Convert frame sequence to spikes using delta modulation.
    Spikes generated when pixel change exceeds threshold.
    
    frames: (T, H, W) - sequence of frames
    Returns: (T, H, W) - binary spike array
    """
    spikes = np.zeros_like(frames)
    
    # Generate spikes for temporal differences
    for t in range(1, len(frames)):
        diff = frames[t] - frames[t-1]
        spikes[t] = (np.abs(diff) > threshold).astype(float)
    
    return spikes

In [4]:
# Example: test with your friend's data generator
# frames = generate_motion_sequence(direction="right", seq_length=30)
# spikes = encode_to_spikes(frames, threshold=0.1)
# print(f"Frame shape: {frames.shape}")
# print(f"Spike shape: {spikes.shape}")
# print(f"Total spikes: {spikes.sum()}")

## Simple SNN (Leaky Integrate-and-Fire)

In [5]:
def lif_neuron(input_spikes, weights, threshold=1.0, decay=0.9):
    """
    Simple LIF neuron that processes spike input.
    
    input_spikes: (T, N) - spike trains over time
    weights: (N,) - synaptic weights
    Returns: output spikes over time
    """
    T = len(input_spikes)
    membrane = 0.0
    output_spikes = []
    
    for t in range(T):
        # Integrate input
        membrane = membrane * decay + np.dot(input_spikes[t], weights)
        
        # Fire if threshold reached
        if membrane >= threshold:
            output_spikes.append(1)
            membrane = 0  # Reset
        else:
            output_spikes.append(0)
    
    return np.array(output_spikes)

In [6]:
# Test LIF neuron
# test_spikes = np.random.rand(50, 10) > 0.8  # Random sparse spikes
# test_weights = np.random.randn(10) * 0.5
# output = lif_neuron(test_spikes, test_weights)
# print(f"Output spikes: {output.sum()} out of {len(output)} timesteps")

## STDP

## TRAINING

In [ ]:
def train_snn(x, y, weights, epochs=3):
    n_samples = x.shape[0]
    pre_trace = np.zeros(1024) # 32 * 32
    post_trace = 0.0

    print("Training started...")
    for epoch in range(epochs):
        for i in range(n_samples):
            if y[i] == 1: # Only training 'Right' detector on 'Right' samples
                # Converting frames to spikes using Delta Modulator logic
                spike_seq = encode_to_spikes(x[i])
                input_sequence = spike_seq.reshape(len(spike_seq), -1)

                membrane = 0.0
                for t in range(len(input_sequence)):
                    # Integrating LIF Neuron Logic
                    membrane = membrane * 0.9 + np.dot(input_sequence[t], weights)

                    output_spike = 0.0
                    if membrane >= 1.0:
                        output_spike = 1.0
                        membrane = 0.0

                    # Calling STDP - Need to update and implement the update_stdp function
                    #weights, pre_trace, post_trace = update_stdp(
                       # weights, pre_trace, post_trace, input_sequence[t], output_spike
                   # )
        print(f"Epoch {epoch+1} complete.")
    return weights